# Mapa de Acesso à Saúde Pública em Campinas

**Projeto Integrador — Bancos de Dados Não Relacionais · PUC-Campinas · 2026**

Autores: Arthur Costa Marques, Conrado Marques, Fábio Barbosa Gonsalez, Patrick Rovaron Franco.
Docente: Joice Marin.

Este notebook lê os dados do projeto (rede de UBS, hospitais e UPAs de Campinas, a partir do CNES e do Censo 2022 do IBGE), mostra os indicadores por região e gera o dashboard.

**Como roda:** por padrão, no modo local, lê os JSONs de `data/`. Não acessa a internet para os dados e não precisa de credenciais. Se a variável de ambiente `MONGO_URI` estiver definida, lê do MongoDB Atlas.

> As etapas 1 a 4 do pipeline original (coleta, tratamento, carga no Atlas) estão em `src/pipeline.py` como **registro histórico** e não são executadas aqui.

## 1. Setup

In [1]:
import os
import runpy
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
import loader

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

# Opcional, só no Colab: usar o secret MONGO_URI em vez dos arquivos locais.
# (Descomente as 2 linhas abaixo. Fora do Colab, ignore.)
# from google.colab import userdata
# os.environ["MONGO_URI"] = userdata.get("MONGO_URI")

print("Fonte de dados:", loader.fonte_em_uso())

Fonte de dados: arquivos locais (data/*.json)


## 2. Dados

`loader.carregar_dados()` devolve dois DataFrames:

- `df_estab`: um estabelecimento por linha (UBS, hospital ou UPA);
- `df_indic`: um distrito de saúde por linha, como gravado em `regioes.json`.

In [2]:
df_estab, df_indic = loader.carregar_dados()
docs = loader.carregar_documentos()

print(f"Estabelecimentos: {len(df_estab)}")
print(f"Regiões:          {len(df_indic)}")
print(f"Relações:         {len(docs['relacoes'])}")
df_estab.head()

Estabelecimentos: 115
Regiões:          5
Relações:         115


,codigo_cnes,nome_fantasia,codigo_tipo_unidade,descricao_tipo_unidade,categoria,bairro_estabelecimento,regiao,latitude,longitude,vinculo_sus,natureza
0,9725407,CENTRO DE SAUDE VICENTE PISANI NETO,2,CENTRO DE SAÚDE / UNIDADE BÁSICA,UBS,SATELITE IRIS II,SUDOESTE,-22.936411,-47.150401,SIM,MUNICIPAL
1,9806342,HOSPITAL SAMARITANO UNIDADE II,5,HOSPITAL GERAL,Hospital,JARDIM NOVA EUROPA,NORTE,-22.931821,-47.057067,NAO,MUNICIPAL
2,2022621,MATERNIDADE DE CAMPINAS,5,HOSPITAL GERAL,Hospital,CENTRO,LESTE,-22.899360,-47.063230,SIM,MUNICIPAL
3,2022702,CENTRO DE SAUDE SOUSAS,2,CENTRO DE SAÚDE / UNIDADE BÁSICA,UBS,SOUSAS,NORTE,-22.876861,-46.969551,SIM,MUNICIPAL
4,2079666,CENTRO MEDICO DE CAMPINAS,5,HOSPITAL GERAL,Hospital,CIDADE UNIVERSITARIA,NORTE,-22.808750,-47.065290,NAO,MUNICIPAL


## 3. Tabelas e indicadores

**Atenção:** `df_indic` traz os indicadores *como foram gravados* na etapa 2 do pipeline, contando **todas** as UBS (inclusive as não-SUS). O dashboard recalcula tudo só com unidades SUS, por isso os valores de hab/UBS abaixo diferem dos do dashboard.

A população por região é uma **estimativa proporcional** (`PROPORCOES_SMS`) aplicada ao total do Censo 2022; o IBGE não publica esse dado por distrito de saúde.

In [3]:
print("Indicadores como gravados em data/regioes.json (todas as UBS):")
df_indic[["regiao", "populacao", "UBS", "Hospital", "UPA", "hab_por_ubs", "cobertura"]]

Indicadores como gravados em data/regioes.json (todas as UBS):


,regiao,populacao,UBS,Hospital,UPA,hab_por_ubs,cobertura
0,NOROESTE,174274,2,1,2,87137.0,Crítica
1,NORTE,191360,9,13,1,21262.0,Crítica
2,SUL,216419,15,8,1,14428.0,Moderada
3,SUDOESTE,342853,26,1,2,13187.0,Moderada
4,LESTE,214141,21,12,1,10197.0,Moderada


In [4]:
# Composição dos 115 estabelecimentos
df_estab["sus"] = df_estab["vinculo_sus"].map({"SIM": "SUS", "NAO": "não-SUS"})
print(pd.crosstab(df_estab["categoria"], df_estab["sus"], margins=True))

# Mesmo filtro do dashboard: UBS/UPA não-SUS saem; hospitais ficam
removidas = df_estab[df_estab["categoria"].isin(["UBS", "UPA"]) & (df_estab["vinculo_sus"] == "NAO")]
sus = df_estab[df_estab["vinculo_sus"] == "SIM"]
print(f"\nUnidades SUS usadas nos indicadores: {len(sus)}  {sus['categoria'].value_counts().to_dict()}")
print(f"UBS/UPA não-SUS removidas pelo dashboard: {len(removidas)}  {removidas['categoria'].value_counts().to_dict()}")
removidas[["categoria", "nome_fantasia", "regiao", "natureza"]].to_string(index=False)

sus        SUS  não-SUS  All
categoria                   
Hospital    16       19   35
UBS         70        3   73
UPA          5        2    7
All         91       24  115

Unidades SUS usadas nos indicadores: 91  {'UBS': 70, 'Hospital': 16, 'UPA': 5}
UBS/UPA não-SUS removidas pelo dashboard: 5  {'UBS': 3, 'UPA': 2}


'categoria                                            nome_fantasia   regiao  natureza\n      UPA                          PRONTO ATENDIMENTO DOCTOR PRIME    NORTE MUNICIPAL\n      UBS CENTRO DE PROGRESSAO PENITENCIARIA PROF ATALIBA NOGUEIRA SUDOESTE  ESTADUAL\n      UBS                CENTRO DE DETENCAO PROVISORIA DE CAMPINAS SUDOESTE  ESTADUAL\n      UPA                          UNIDADE AVANCADA MADRE THEODORA    LESTE MUNICIPAL\n      UBS                                           CASA JEQUITIBA    LESTE MUNICIPAL'

In [5]:
# Indicadores recalculados só com unidades SUS (mesma conta do dashboard: população / nº de UBS SUS)
pivot = sus.groupby(["regiao", "categoria"]).size().unstack(fill_value=0)
ind_sus = df_indic[["regiao", "populacao"]].merge(pivot, on="regiao", how="left").fillna(0)
ind_sus["hab_por_ubs_SUS"] = (ind_sus["populacao"] / ind_sus["UBS"]).round(0)
ind_sus.sort_values("hab_por_ubs_SUS", ascending=False).reset_index(drop=True)

,regiao,populacao,Hospital,UBS,UPA,hab_por_ubs_SUS
0,NOROESTE,174274,1,2,2,87137.0
1,NORTE,191360,4,9,0,21262.0
2,SUL,216419,6,15,1,14428.0
3,SUDOESTE,342853,1,24,2,14286.0
4,LESTE,214141,4,20,0,10707.0


## 4. Dashboard (v4)

Executa `src/dashboard.py`, que lê os mesmos dados pelo `loader` e gera um HTML. O arquivo vai para `build/dashboard_gerado.html` (ignorado pelo git). O dashboard publicado do projeto é `docs/index.html`.

O HTML carrega Plotly, Leaflet, Google Fonts e tiles do CARTO por CDN, então precisa de internet para ser aberto.

In [6]:
os.environ["DASHBOARD_SAIDA"] = str(RAIZ / "build" / "dashboard_gerado.html")
_ = runpy.run_path(str(RAIZ / "src" / "dashboard.py"))

📡 Carregando dados de: arquivos locais (data/*.json)
  ✓ 115 estabelecimentos carregados
  ✓ 5 UBS/UPA não-SUS removidas (artefatos)
  ✓ 110 estabelecimentos válidos para o painel



✓ Dashboard v4 salvo: build\dashboard_gerado.html
  Tamanho: 108,774 bytes


In [7]:
# Confere: o texto do dashboard gerado agora é o mesmo do docs/index.html?
import re

def texto_visivel(caminho):
    h = Path(caminho).read_text(encoding="utf-8")
    h = re.sub(r"<script.*?</script>|<style.*?</style>", "", h, flags=re.S)
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", h))

gerado = texto_visivel(RAIZ / "build" / "dashboard_gerado.html")
publicado = texto_visivel(RAIZ / "docs" / "index.html")
print("Texto visível idêntico ao docs/index.html:", gerado == publicado)

Texto visível idêntico ao docs/index.html: True
